In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 14 — Local Financial Report Analyst
## QA Over Annual Reports with Table + Text Parsing

**Stack:** Ollama · LangChain · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb pandas

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.1)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

## Step 2 — Create Sample Financial Data

In [ ]:
from pathlib import Path
Path("sample_data").mkdir(exist_ok=True)

# Simulated financial statements
income_data = {
    "Year": [2022, 2023, 2024],
    "Revenue_M": [450, 580, 720],
    "COGS_M": [180, 220, 270],
    "Gross_Profit_M": [270, 360, 450],
    "Operating_Expenses_M": [150, 180, 210],
    "Net_Income_M": [85, 128, 175],
}
df = pd.DataFrame(income_data)
df.to_csv("sample_data/income_statement.csv", index=False)
print(df.to_string(index=False))

narrative = """
Annual Report 2024 — TechCorp Inc.

Financial Highlights:
Revenue grew 24% year-over-year to $720M, driven by enterprise SaaS expansion.
Gross margins improved to 62.5% from 62.1% as we optimized cloud infrastructure.
Net income reached $175M, a 37% increase, reflecting operating leverage.

Strategic Initiatives:
- Launched AI-powered analytics platform (Q2 2024), contributing $45M in new ARR
- Expanded to 3 new international markets (Germany, Japan, Brazil)
- Acquired DataViz Corp for $50M to enhance visualization capabilities

Risk Factors:
- Enterprise sales cycles lengthening from 45 to 60 days
- Currency headwinds expected to impact 2025 revenue by 2-3%
- Key competitor launched similar AI features at lower price point
"""
Path("sample_data/annual_report.txt").write_text(narrative, encoding="utf-8")
print("\nFinancial data and narrative report created.")

## Step 3 — Build Combined Text + Table Index

In [ ]:
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# Load narrative
narrative_doc = Document(
    page_content=Path("sample_data/annual_report.txt").read_text(encoding="utf-8"),
    metadata={"type": "narrative", "source": "annual_report"}
)

# Convert table to natural language for indexing
table_text = "Income Statement Summary:\n"
for _, row in df.iterrows():
    table_text += (f"Year {int(row['Year'])}: Revenue ${row['Revenue_M']}M, "
                  f"Gross Profit ${row['Gross_Profit_M']}M "
                  f"({row['Gross_Profit_M']/row['Revenue_M']*100:.1f}% margin), "
                  f"Net Income ${row['Net_Income_M']}M\n")

table_doc = Document(
    page_content=table_text,
    metadata={"type": "financial_table", "source": "income_statement"}
)

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
all_chunks = splitter.split_documents([narrative_doc, table_doc])

vectorstore = Chroma.from_documents(all_chunks, embeddings,
    persist_directory="sample_data/financial_chroma", collection_name="financial")
print(f"Indexed {len(all_chunks)} chunks (narrative + tables)")

## Step 4 — Financial Q&A

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

fin_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a financial analyst assistant. Answer questions about the
company using the provided financial data and reports.

When citing numbers, be precise and show calculations where applicable.

Data:
{context}

Question: {question}

Analysis:"""
)

fin_qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": fin_prompt},
)

questions = [
    "What was the revenue growth rate from 2023 to 2024?",
    "What are the main risk factors for 2025?",
    "How much did the AI analytics platform contribute in ARR?",
    "Calculate the net income margin trend over 3 years.",
]

for q in questions:
    print(f"\nQ: {q}")
    result = fin_qa.invoke({"query": q})
    print(f"A: {result['result']}")

## Step 5 — Trend Analysis with pandas

In [ ]:
# Compute key metrics
df["Gross_Margin_Pct"] = (df["Gross_Profit_M"] / df["Revenue_M"] * 100).round(1)
df["Net_Margin_Pct"] = (df["Net_Income_M"] / df["Revenue_M"] * 100).round(1)
df["Revenue_Growth_Pct"] = df["Revenue_M"].pct_change().mul(100).round(1)

print("Key Financial Metrics:")
print(df[["Year", "Revenue_M", "Gross_Margin_Pct", "Net_Margin_Pct", "Revenue_Growth_Pct"]].to_string(index=False))

# Generate natural language summary of trends
trend_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a financial analyst. Summarize key trends from the data."),
    ("human", "Analyze these trends:\n{data}")
])
trend_chain = trend_prompt | llm | StrOutputParser()
analysis = trend_chain.invoke({"data": df.to_string()})
print(f"\nTrend Analysis:\n{analysis}")

## What You Learned
- **Combined table + text RAG** for financial analysis
- **Table-to-text conversion** for indexing structured data
- **Financial metrics computation** with pandas
- **Trend analysis** combining quantitative and qualitative data